In [1]:
from datasets import load_dataset
import pandas as pd

In [2]:
train_data = load_dataset("pile-of-law/pile-of-law", 'r_legaladvice', split="train")
validation_data = load_dataset("pile-of-law/pile-of-law", 'r_legaladvice', split="validation")

Found cached dataset pile-of-law (C:/Users/Ali/.cache/huggingface/datasets/pile-of-law___pile-of-law/r_legaladvice/0.0.0/c1090502f95031ebfad49ede680394da5532909fa46b7a0452be8cddecc9fa60)
Found cached dataset pile-of-law (C:/Users/Ali/.cache/huggingface/datasets/pile-of-law___pile-of-law/r_legaladvice/0.0.0/c1090502f95031ebfad49ede680394da5532909fa46b7a0452be8cddecc9fa60)


In [21]:
train_set = pd.DataFrame(train_data, columns=["text", "created_timestamp", "downloaded_timestamp", "url"])
validation_set = pd.DataFrame(validation_data, columns=["text", "created_timestamp", "downloaded_timestamp", "url"])

In [22]:
df = pd.concat([train_set, validation_set])

In [23]:
df = df.drop(['created_timestamp','downloaded_timestamp'], axis=1)

In [24]:
df["text"] = df["text"].str.replace("\n","")

In [25]:
df["Title"] = df['text'].apply(lambda x: x[7:x.find("Question")])

In [26]:
df["Question"] = df['text'].apply(lambda x: x[x.find("Question")+9:x.find("Answer")])

In [27]:
df["Answer"] = df["text"].apply(lambda x: x[x.find("Answer")+11:])

In [113]:
df.drop(["text"], axis=1, inplace=True)

In [ ]:
from datasets import load_dataset
import pandas as pd

train_data = load_dataset("pile-of-law/pile-of-law", 'r_legaladvice', split="train")
validation_data = load_dataset("pile-of-law/pile-of-law", 'r_legaladvice', split="validation")

train_set = pd.DataFrame(train_data, columns=["text", "created_timestamp", "downloaded_timestamp", "url"])
validation_set = pd.DataFrame(validation_data, columns=["text", "created_timestamp", "downloaded_timestamp", "url"])

df = pd.concat([train_set, validation_set])
df = df.drop(['created_timestamp','downloaded_timestamp'], axis=1)
df["text"] = df["text"].str.replace("\n","")
df["Title"] = df['text'].apply(lambda x: x[7:x.find("Question")])
df["Question"] = df['text'].apply(lambda x: x[x.find("Question")+9:x.find("Answer")])
df["Answer"] = df["text"].apply(lambda x: x[x.find("Answer")+11:])
df.drop(["text"], axis=1, inplace=True)

In [17]:
df.index = range(0, df.shape[0])

In [28]:
df.index()

TypeError: 'Index' object is not callable

In [35]:
df.columns = ["Text", "Url", "Title", "Question", "Answer"]

In [36]:
df

,Text,Url,Title,Question,Answer
0,"Title: Landlord broke lease agreement, what ar...",https://www.reddit.com/r/legaladvice/comments/...,"Landlord broke lease agreement, what are my ri...",Our landlord has been promising us a washer/dr...,You can let your landlord know in writing that...
1,Title: I think someone is breaking into my car...,https://www.reddit.com/r/legaladvice/comments/...,I think someone is breaking into my car to get...,"This is in Ohio. Back in May, I accidentally d...",File a police report for every break in and ge...
2,Title: MA - Just found out I might not be able...,https://www.reddit.com/r/legaladvice/comments/...,MA - Just found out I might not be able to bui...,Two years ago I bought a piece of land in Mass...,"I kept reading and reading, and kept thinking ..."
3,Title: I adopted a dog. They said they gave me...,https://www.reddit.com/r/legaladvice/comments/...,I adopted a dog. They said they gave me the wr...,I recently adopted a puppy from a humane socie...,I haven’t seen this mentioned yet but I would ...
4,Title: (New Jersey) Denied haircut at a Barber...,https://www.reddit.com/r/legaladvice/comments/...,(New Jersey) Denied haircut at a Barbershop fo...,I’ll keep this as short and to the point as po...,"Not sure about NJ, but there was a case in my ..."
...,...,...,...,...,...
36926,Title: habitual Traffic Offender. Got caught a...,https://www.reddit.com/r/legaladvice/comments/...,habitual Traffic Offender. Got caught again. j...,I live in panhandle Florida. I got a letter in...,Poor decision after poor decision has finally ...
36927,Title: I am being charged with Robbery in CA. ...,https://www.reddit.com/r/legaladvice/comments/...,I am being charged with Robbery in CA. Need ad...,[deleted],You need a defense attorney. It's possible tha...
36928,"Title: [Philadelphia,PA] Late diagnosis of a b...",http://www.reddit.com/r/legaladvice/comments/2...,"[Philadelphia,PA] Late diagnosis of a benign b...","Hi! First and foremost, thanks for reading t...","TL;DR: OP had chronic pain, and after first M..."
36929,Title: [Florida] My Girlfriend's admitted to m...,https://www.reddit.com/r/legaladvice/comments/...,[Florida] My Girlfriend's admitted to me that ...,My (M22) Girlfirend (F18) admitted to me a whi...,"I am not 100% sure about FL, but I assume that..."


In [42]:
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader, DataFrameLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma

In [43]:
loader = DataFrameLoader(df, page_content_column="Text")

In [45]:
documents = loader.load()

In [49]:
len(df)

146671

[Document(page_content='Title: Landlord broke lease agreement, what are my rights? (Chicago, IL)Question:Our landlord has been promising us a washer/dryer unit since we moved in (July 2015). When we resigned the lease August 2016, we wrote into the lease that an in-unit washer and dryer would be installed by September 30th 2016.Since September 30th, there have been continuous delays in getting the W/D installed. Since it has now been almost a month past the date the W/D was supposed to be installed, I am wondering what types of rights as a tenant I have? Thanks ahead of time for any and all advice given.Answer #1: You can let your landlord know in writing that he is in default under the current lease agreement and give him a reasonable timeframe to cure his default.  If he fails to correct the default, you can likely end your lease and move.', metadata={'Url': 'https://www.reddit.com/r/legaladvice/comments/59cv5x/landlord_broke_lease_agreement_what_are_my_rights/', 'Title': 'Landlord b

In [50]:
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)